<a href="https://colab.research.google.com/github/vedantdharme/Data_science_lab_SE_A_13/blob/main/experiment1_A__algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### A* Algorithm Implementation

First, let's define a `Node` class to represent the states in our search space. Each node will store its state, its parent node, the cost to reach this node (`g_cost`), and its heuristic cost (`h_cost`).

In [1]:
import heapq

class Node:
    def __init__(self, state, parent=None, g_cost=0, h_cost=0):
        self.state = state
        self.parent = parent
        self.g_cost = g_cost  # Cost from start node to current node
        self.h_cost = h_cost  # Estimated cost from current node to goal node
        self.f_cost = g_cost + h_cost # Total estimated cost

    def __lt__(self, other):
        return self.f_cost < other.f_cost

    def __eq__(self, other):
        return self.state == other.state

    def __hash__(self):
        return hash(self.state)

    def __repr__(self):
        return f"Node(state={self.state}, f={self.f_cost:.2f}, g={self.g_cost:.2f}, h={self.h_cost:.2f})"


Next, we'll define a simple heuristic function. For this example, we'll use a placeholder heuristic that always returns 0 (which means it degenerates to Dijkstra's algorithm). In a real application, this would be problem-specific (e.g., Manhattan distance for a grid, Euclidean distance for coordinates).

In [2]:
def heuristic(state, goal_state):
    """A placeholder heuristic function.
    In a real-world problem, this would estimate the cost from the current state to the goal.
    For simplicity, we return 0, making A* behave like Dijkstra's algorithm.
    """
    # Example: For a 2D grid, this could be Manhattan distance:
    # return abs(state[0] - goal_state[0]) + abs(state[1] - goal_state[1])
    return 0


Now, let's implement the A* search algorithm. This function will take the start state, goal state, a function to generate neighbors, and the heuristic function as input.

In [3]:
def a_star_search(start_state, goal_state, get_neighbors_fn, heuristic_fn):
    """
    Implements the A* search algorithm.

    Args:
        start_state: The initial state.
        goal_state: The target state.
        get_neighbors_fn: A function that takes a state and returns a list of (neighbor_state, cost_to_neighbor) tuples.
        heuristic_fn: A function that takes a state and the goal_state and returns an estimated cost to the goal.

    Returns:
        A list of states representing the path from start to goal, or None if no path is found.
    """
    # Use a priority queue to store nodes to be explored, ordered by f_cost
    open_list = []
    start_node = Node(start_state, g_cost=0, h_cost=heuristic_fn(start_state, goal_state))
    heapq.heappush(open_list, start_node)

    # Keep track of nodes that have already been explored
    closed_set = set()

    # Keep track of the best g_cost found so far for each state
    g_costs = {start_state: 0}

    # Keep track of the actual nodes to reconstruct the path
    came_from = {}

    while open_list:
        current_node = heapq.heappop(open_list)

        if current_node.state == goal_state:
            path = []
            while current_node:
                path.append(current_node.state)
                current_node = current_node.parent
            return path[::-1] # Return reversed path

        closed_set.add(current_node.state)

        for neighbor_state, cost_to_neighbor in get_neighbors_fn(current_node.state):
            if neighbor_state in closed_set:
                continue

            tentative_g_cost = current_node.g_cost + cost_to_neighbor

            # If we found a shorter path to the neighbor or haven't seen it yet
            if neighbor_state not in g_costs or tentative_g_cost < g_costs[neighbor_state]:
                g_costs[neighbor_state] = tentative_g_cost
                h_cost = heuristic_fn(neighbor_state, goal_state)
                neighbor_node = Node(neighbor_state, current_node, tentative_g_cost, h_cost)
                heapq.heappush(open_list, neighbor_node)
                came_from[neighbor_state] = current_node

    return None # No path found


### Example Usage

Let's define a simple graph and use the A* algorithm to find a path.

Consider a graph represented as an adjacency list:
```
  A --(1)--> B --(3)--> D
  |           |         ^
 (2)         (1)       (2)
  v           v         |
  C --(1)--> E --(1)--> F
```
Goal: A to F

In [4]:
# Define the graph as an adjacency list with costs
graph = {
    'A': [('B', 1), ('C', 2)],
    'B': [('D', 3), ('E', 1)],
    'C': [('E', 1)],
    'D': [('F', 2)],
    'E': [('F', 1)],
    'F': []
}

def get_graph_neighbors(state):
    """Returns neighbors for a given state in our example graph."""
    return graph.get(state, [])

# Define the start and goal states
start = 'A'
goal = 'F'

# Run A* search
path = a_star_search(start, goal, get_graph_neighbors, heuristic)

if path:
    print(f"Path from {start} to {goal}: {path}")
else:
    print(f"No path found from {start} to {goal}")


Path from A to F: ['A', 'B', 'E', 'F']


### Explanation of the Output

The output shows the path found by the A* algorithm. Since our `heuristic` function currently returns 0, this implementation effectively performs Dijkstra's algorithm, finding the shortest path based solely on the cumulative cost (`g_cost`).

If you were to implement a proper admissible and consistent heuristic (e.g., for a grid-based pathfinding where you know the straight-line distance to the goal), A* would typically find the path more efficiently by prioritizing nodes that seem closer to the goal.